# <font color="#418FDE" size="6.5" uppercase>**Roots and Iteration**</font>

>Last update: 20260606.
    
By the end of this Lecture, you will be able to:
- Implement iterative algorithms to approximate roots of engineering equations. 
- Use tolerances and stopping criteria to control numerical solution quality. 
- Assess convergence behavior using iteration counts and residual values. 


## **1. Bisection Method**

### **1.1. Starting Interval**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Engineers/Module_04/Lecture_A/image_01_01.jpg?v=1780719824" width="250">



>* Choose endpoints with opposite function signs
>* Bracket the root for reliable searching

>* Use engineering limits to choose intervals
>* Bracket the root without wasting iterations

>* Check endpoints for a sign change
>* Use realistic intervals to bound the root



In [ ]:
#@title Python Code - Starting Interval

# This script checks bisection starting intervals.
# Opposite signs suggest a trapped root.
# Printed results stay short and focused.

# Define an engineering balance function.
def residual(flow_rate):
    pressure_available = 120.0
    pressure_loss = 4.0 * flow_rate**2
    return pressure_available - pressure_loss

# Store candidate starting intervals.
intervals = [(1.0, 3.0), (3.0, 7.0), (7.0, 9.0)]
valid_intervals = []

# Check endpoints for sign changes.
for lower, upper in intervals:
    f_lower = residual(lower)
    f_upper = residual(upper)

    # A negative product means opposite signs.
    has_sign_change = f_lower * f_upper < 0.0
    if has_sign_change:
        valid_intervals.append((lower, upper))

    # Print one compact diagnostic line.
    status = "valid" if has_sign_change else "not valid"
    print(f"[{lower:.1f}, {upper:.1f}] gives {status} bracket")

# Select the first valid starting interval.
start_interval = valid_intervals[0] if valid_intervals else None

# Report the chosen interval for bisection.
message = f"Chosen starting interval: {start_interval}"
print(message)



### **1.2. Midpoint Update**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Engineers/Module_04/Lecture_A/image_01_02.jpg?v=1780719826" width="250">



>* Test the function at the interval midpoint
>* Keep the half that still brackets the root

>* Test the midpoint to narrow uncertainty
>* Apply the same logic across engineering models

>* Reliable halving improves difficult root estimates
>* Maintain sign bracketing when replacing endpoints



In [ ]:
#@title Python Code - Midpoint Update

# Demonstrate midpoint updates for bisection.
# Track interval shrinking and residuals.
# Use a simple engineering equation.

import math

# Define a continuous pressure balance equation.
def balance_error(flow_rate):
    return flow_rate**3 - flow_rate - 2

# Store the initial bracketing interval.
left_value = 1.0
right_value = 2.0

# Check that the root is bracketed.
left_error = balance_error(left_value)
right_error = balance_error(right_value)

# Stop early if signs do not bracket.
if left_error * right_error >= 0:
    raise ValueError("Initial interval does not bracket a root")

# Choose a small fixed iteration count.
max_updates = 6
records = []

# Repeatedly compute and test the midpoint.
for step_index in range(1, max_updates + 1):
    midpoint = (left_value + right_value) / 2.0
    midpoint_error = balance_error(midpoint)

    # Save the current midpoint information.
    interval_width = right_value - left_value
    records.append((step_index, midpoint, midpoint_error, interval_width))

    # Replace the endpoint sharing the midpoint sign.
    if left_error * midpoint_error <= 0:
        right_value = midpoint
        right_error = midpoint_error
    else:
        left_value = midpoint
        left_error = midpoint_error

# Print a compact convergence table.
print("Bisection midpoint update example")
print("step | midpoint | residual | interval width")

# Show each saved update in one line.
for step_index, midpoint, residual, width in records:
    print(f"{step_index:>4} | {midpoint:>8.5f} | {residual:>8.5f} | {width:>13.5f}")

# Report the final bracketing interval.
final_midpoint = (left_value + right_value) / 2.0
print(f"Final bracket: [{left_value:.5f}, {right_value:.5f}]")
print(f"Final midpoint estimate: {final_midpoint:.5f}")



### **1.3. Root Approximation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Engineers/Module_04/Lecture_A/image_01_03.jpg?v=1780719795" width="250">



>* Midpoint estimates the root within the interval
>* Each step halves the remaining uncertainty

>* Midpoint estimates have clear engineering meaning
>* Remaining interval shows current root uncertainty

>* Refine estimates without expecting exact roots
>* Stop when accuracy fits the engineering need



In [ ]:
#@title Python Code - Root Approximation

# Bisection narrows intervals around roots.
# Midpoints provide practical engineering approximations.
# Residuals show approximation quality.

# Define a simple engineering balance function.
def force_balance(pressure_value):

    return pressure_value**2 - 30.0

# Choose endpoints with opposite function signs.
left_value = 5.0
right_value = 6.0

# Check that bisection is appropriate here.
left_sign = force_balance(left_value)
right_sign = force_balance(right_value)

# Stop when interval width becomes sufficiently small.
tolerance_value = 0.001
maximum_iterations = 20

# Store a compact iteration history for teaching.
history_rows = []
root_estimate = None

# Repeatedly halve the interval containing the root.
for iteration_number in range(1, maximum_iterations + 1):

    midpoint_value = (left_value + right_value) / 2.0
    residual_value = force_balance(midpoint_value)

    interval_width = right_value - left_value
    history_rows.append((iteration_number, midpoint_value, residual_value, interval_width))

    if abs(residual_value) < tolerance_value:
        root_estimate = midpoint_value
        break

    if abs(interval_width) < tolerance_value:
        root_estimate = midpoint_value
        break

    if left_sign * residual_value < 0.0:
        right_value = midpoint_value
        right_sign = residual_value

    else:
        left_value = midpoint_value
        left_sign = residual_value

# Use latest midpoint if loop ended by iteration limit.
if root_estimate is None:
    root_estimate = history_rows[-1][1]

# Print a short readable convergence summary.
print("Bisection root approximation example")
print(f"Initial pressure interval: [{5.0}, {6.0}]")
print(f"Final pressure estimate: {root_estimate:.6f}")
print(f"Final residual value: {force_balance(root_estimate):.6e}")

# Show selected iterations without excessive output.
print("Iteration | midpoint | residual | interval width")
for row_values in history_rows[:3] + history_rows[-3:]:

    iter_num, mid_val, res_val, width_val = row_values
    print(f"{iter_num:9d} | {mid_val:8.5f} | {res_val:8.3e} | {width_val:8.5f}")



## **2. Fixed Iteration**

### **2.1. Iteration Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Engineers/Module_04/Lecture_A/image_02_01.jpg?v=1780719828" width="250">



>* Caps repeated root-finding updates
>* Stops runaway calculations when convergence fails

>* Balance convergence chances against wasted computation
>* Limit difficult cases to protect simulations

>* Iteration limit does not prove convergence
>* Check residuals and estimate changes



In [ ]:
#@title Python Code - Iteration Limits

# Fixed iterations can stop unsuccessful searches.
# Residuals reveal whether estimates are acceptable.
# Compare several limits using bisection.

def f(x):
    return x**3 - x - 2

# Store a valid initial bracket.
left_start, right_start = 1.0, 2.0

# Verify the bracket contains sign change.
if f(left_start) * f(right_start) >= 0:
    raise ValueError("Initial bracket has no sign change.")

# Set several allowed iteration counts.
limits = [2, 5, 10, 20]
tolerance = 1e-6

# Run bisection with each iteration limit.
results = []
for limit in limits:
    left, right = left_start, right_start

    # Repeat only until the fixed limit.
    for iteration in range(1, limit + 1):
        midpoint = (left + right) / 2
        residual = abs(f(midpoint))

        # Stop early if tolerance is satisfied.
        if residual < tolerance:
            break
        if f(left) * f(midpoint) < 0:
            right = midpoint

        else:
            left = midpoint

    # Save final estimate and quality information.
    results.append((limit, iteration, midpoint, residual))

# Display compact comparison table.
print("limit | used | estimate  | residual")
for limit, used, estimate, residual in results:
    print(f"{limit:5d} | {used:4d} | {estimate:9.6f} | {residual:.2e}")

# Emphasize the main numerical lesson.
print("A limit stops work, but residual checks quality.")



### **2.2. Tolerance Based Stopping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Engineers/Module_04/Lecture_A/image_02_02.jpg?v=1780719791" width="250">



>* Stop when the root estimate is close enough
>* Avoid wasted computation beyond practical accuracy

>* Match tolerance to method and context
>* Balance accuracy needs against wasted computation

>* Use multiple checks to judge convergence
>* Interpret tolerances with problem context



In [ ]:
#@title Python Code - Tolerance Based Stopping

# Demonstrate tolerance based stopping.
# Compare residual and change checks.
# Keep output short and readable.

# Define the engineering equation.
def balance_function(flow_rate):
    return flow_rate**2 - 10.0

# Set starting guess and tolerances.
estimate = 1.0
tolerance_change = 1e-6
tolerance_residual = 1e-6
max_iterations = 30

# Track the iteration history.
history = []
stopped_reason = "maximum iterations reached"
last_change = float("inf")

# Apply Newton iteration safely.
for iteration in range(1, max_iterations + 1):
    derivative = 2.0 * estimate
    if derivative == 0.0:
        stopped_reason = "zero derivative encountered"
        break

    next_estimate = estimate - balance_function(estimate) / derivative
    last_change = abs(next_estimate - estimate)
    residual = abs(balance_function(next_estimate))
    history.append((iteration, next_estimate, last_change, residual))

    if last_change < tolerance_change and residual < tolerance_residual:
        stopped_reason = "both tolerances satisfied"
        estimate = next_estimate
        break

    estimate = next_estimate

# Extract the final stored result.
final_iteration, final_root, final_change, final_residual = history[-1]

# Print a compact convergence summary.
print("Tolerance based stopping for sqrt(10).")
print(f"Stop reason: {stopped_reason}.")
print(f"Iterations used: {final_iteration}.")
print(f"Estimated root: {final_root:.8f}.")
print(f"Last change: {final_change:.2e}.")
print(f"Final residual: {final_residual:.2e}.")

# Show selected iteration details.
print("Iter | estimate | change | residual")
for row in history[:3] + history[-2:]:
    print(f"{row[0]:>4} | {row[1]:.6f} | {row[2]:.1e} | {row[3]:.1e}")



### **2.3. Stopping Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Engineers/Module_04/Lecture_A/image_02_03.jpg?v=1780719790" width="250">



>* Stop when the approximation is good enough
>* Rules prevent endless, unreliable iteration

>* Match stopping rules to problem meaning
>* Check both root changes and residuals

>* Combine criteria for accuracy, efficiency, and safety
>* Choose tolerances appropriate for engineering decisions



In [ ]:
#@title Python Code - Stopping Criteria

# Demonstrate stopping criteria for fixed-point iteration.
# Compare change tolerance and residual tolerance.
# Limit iterations to protect computation.

# Define an engineering-style nonlinear balance function.
def balance_residual(x):
    return x * x * x - x - 2

# Define a simple fixed-point update formula.
def fixed_point_update(x):
    return (x + 2) ** (1 / 3)

# Set stopping tolerances and iteration limit.
x_old = 1.0
tol_change = 1e-6
tol_residual = 1e-6
max_iterations = 50

# Store final status for concise reporting.
stop_reason = "maximum iterations reached"
iterations_used = 0

# Iterate until any stopping rule is satisfied.
for iteration in range(1, max_iterations + 1):
    x_new = fixed_point_update(x_old)
    change = abs(x_new - x_old)
    residual = abs(balance_residual(x_new))

    # Save current iteration count.
    iterations_used = iteration
    if change < tol_change and residual < tol_residual:
        stop_reason = "both tolerances satisfied"
        break

    # Continue from the newest approximation.
    x_old = x_new

# Print a compact summary of solution quality.
print("Approximate root:", round(x_new, 8))
print("Iterations used:", iterations_used)
print("Final change:", format(change, ".2e"))
print("Final residual:", format(residual, ".2e"))
print("Stopping reason:", stop_reason)



## **3. Convergence Checks**

### **3.1. Residual Trends**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Engineers/Module_04/Lecture_A/image_03_01.jpg?v=1780719783" width="250">



>* Residuals measure how well estimates satisfy equations
>* Decreasing residuals suggest convergence toward the root

>* Track residuals across all iterations
>* Watch for slowing, oscillating, or increasing residuals

>* Judge residual size within practical context
>* Compare trends with iterations and system knowledge



In [ ]:
#@title Python Code - Residual Trends

# Residual trends reveal iterative root progress.
# This example uses simple bisection iteration.
# Printed residuals stay short and readable.

import math
import matplotlib.pyplot as plt

# Define an engineering-style balance equation.
def f(x):
    return x**3 - x - 2

# Store iteration numbers and residual magnitudes.
iterations = []
residuals = []

# Choose a bracket containing one root.
left, right = 1.0, 2.0

# Use a practical residual tolerance.
tolerance = 1e-6

# Limit iterations for safe execution.
max_steps = 25

# Confirm that bisection is valid here.
if f(left) * f(right) > 0:
    raise ValueError("Bracket does not contain a sign change")

# Run bisection and record residual trend.
for step in range(1, max_steps + 1):
    midpoint = (left + right) / 2
    residual = abs(f(midpoint))

    iterations.append(step)
    residuals.append(residual)

    if residual < tolerance:
        break

    if f(left) * f(midpoint) < 0:
        right = midpoint
    else:
        left = midpoint

# Print a compact convergence summary.
print(f"Iterations completed: {iterations[-1]}")
print(f"Approximate root: {midpoint:.8f}")
print(f"Final residual: {residuals[-1]:.2e}")
print("Residual trend: decreasing overall")

# Plot residual magnitude versus iteration number.
plt.figure(figsize=(6, 4))
plt.semilogy(iterations, residuals, marker="o")
plt.xlabel("Iteration number")
plt.ylabel("Absolute residual |f(x)|")
plt.title("Residual Trend During Bisection")
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()



### **3.2. Iteration Trends**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Engineers/Module_04/Lecture_A/image_03_02.jpg?v=1780719784" width="250">



>* Track how approximations move toward roots
>* Iteration counts reveal method and guess quality

>* Steady improvements show healthy convergence
>* Erratic iterations signal method or model issues

>* Judge iterations with residuals and problem context
>* Balance accuracy, cost, speed, and reliability



In [ ]:
#@title Python Code - Iteration Trends

# Compare iteration histories for two guesses.
# Residuals reveal convergence quality clearly.
# Small tables keep output readable.

import math
import matplotlib.pyplot as plt

# Define a nonlinear engineering equation.
def residual(flow_rate):
    return flow_rate**3 - 6.0 * flow_rate - 4.0

# Build Newton iteration history safely.
def newton_history(start_value, max_steps, tolerance):
    history = []
    current_value = float(start_value)

    for step_index in range(1, max_steps + 1):
        current_residual = residual(current_value)
        history.append((step_index, current_value, abs(current_residual)))

        derivative_value = 3.0 * current_value**2 - 6.0
        if abs(current_residual) < tolerance or abs(derivative_value) < 1e-12:
            break

        current_value = current_value - current_residual / derivative_value
    return history

# Compute two histories from different guesses.
good_history = newton_history(3.0, 12, 1e-8)
slow_history = newton_history(1.6, 12, 1e-8)

# Print a compact convergence summary.
print("Guess  Iterations  Final x      Final residual")
for label, history in [("3.0", good_history), ("1.6", slow_history)]:
    final_step, final_x, final_residual = history[-1]

    print(f"{label:>4}  {final_step:>10}  {final_x:>9.5f}  {final_residual:>14.2e}")

# Plot residual decrease across iterations.
for label, history in [("start 3.0", good_history), ("start 1.6", slow_history)]:
    steps = [row[0] for row in history]

    residuals = [row[2] for row in history]
    plt.semilogy(steps, residuals, marker="o", label=label)

# Label the iteration trend plot.
plt.xlabel("Iteration count")
plt.ylabel("Absolute residual")
plt.title("Iteration trends for Newton root finding")
plt.grid(True, which="both", linestyle="--", alpha=0.5)

# Show the comparison legend.
plt.legend()
plt.tight_layout()
plt.show()



### **3.3. Error Trends**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Engineers/Module_04/Lecture_A/image_03_03.jpg?v=1780719778" width="250">



>* Compare successive estimates to track error
>* Shrinking errors suggest convergence toward a root

>* Small errors can hide poor solutions
>* Check error trends with residuals

>* Error decrease rate shows method efficiency.
>* Trends help diagnose limits and suitability.



In [ ]:
#@title Python Code - Error Trends

# Compare error trends during iteration.
# Residuals confirm equation satisfaction.
# Plots reveal convergence behavior.

import math
import matplotlib.pyplot as plt

# Define an engineering-style nonlinear equation.
def function_value(flow_rate):
    return flow_rate**3 - flow_rate - 2

# Store Newton iteration history.
x_values = [1.5]
errors = [math.nan]
residuals = [abs(function_value(x_values[0]))]

# Run a small fixed iteration count.
for step_index in range(1, 8):
    current_x = x_values[-1]
    derivative = 3 * current_x**2 - 1
    next_x = current_x - function_value(current_x) / derivative

    # Track change and residual each iteration.
    x_values.append(next_x)
    errors.append(abs(next_x - current_x))
    residuals.append(abs(function_value(next_x)))

# Print compact convergence table.
print("iter | estimate | error | residual")
for index, estimate in enumerate(x_values):
    error_text = "start" if index == 0 else f"{errors[index]:.2e}"
    print(f"{index:4d} | {estimate:8.5f} | {error_text:>8} | {residuals[index]:.2e}")

# Plot error and residual trends together.
iteration_numbers = list(range(len(x_values)))
plt.figure(figsize=(7, 4))
plt.semilogy(iteration_numbers[1:], errors[1:], "o-", label="estimated error")
plt.semilogy(iteration_numbers, residuals, "s--", label="residual")

# Label axes and explain the comparison.
plt.xlabel("Iteration number")
plt.ylabel("Magnitude on log scale")
plt.title("Error trend versus residual trend")
plt.grid(True, which="both", linestyle=":")

# Display the single convergence plot.
plt.legend()
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Roots and Iteration**</font>


In this lecture, you learned to:
- Implement iterative algorithms to approximate roots of engineering equations. 
- Use tolerances and stopping criteria to control numerical solution quality. 
- Assess convergence behavior using iteration counts and residual values. 

In the next Lecture (Lecture B), we will go over 'Integration and Simulation'